In [ ]:
'''
My first attempt at building a CNN to recognize and classify images.
We will try to classify what game someone is playing from a kaggle dataset of 
game screenshots, https://www.kaggle.com/datasets/aditmagotra/gameplay-images .
There are 1,000 screenshots each from 10 different popular games.
'''

from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.dataset import random_split
from torchvision import datasets, utils
from torchvision import transforms
from torchvision.io import decode_image
import torch.nn.functional as F
import kagglehub


In [ ]:
gamedata_path = kagglehub.dataset_download("aditmagotra/gameplay-images")

In [ ]:
gen1 = torch.Generator().manual_seed(42)
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(device)

In [ ]:
target_lambda = transforms.v2.Lambda(lambda x: F.one_hot(torch.tensor(x), num_classes = 10).float())

# train_transform = transforms.v2.Compose([
#     transforms.v2.ToImage(), 
#     transforms.ColorJitter(brightness = .2, contrast = .2, saturation = .2, hue = .4),
#     transforms.v2.ToDtype(torch.float32, scale = True)
# ])

#Don't have code for splitting with proprietary transforms at this moment.

test_transform = transforms.v2.Compose([
    transforms.v2.ToImage(),
    transforms.v2.ToDtype(torch.float32, scale = True)
])

In [ ]:
full = datasets.ImageFolder(root = gamedata_path, transform= test_transform, target_transform = target_lambda)
test, train = random_split(full, (.2,.8), generator = gen1)

test_loader = DataLoader(test, batch_size = 32, shuffle = False)
train_loader = DataLoader(train, batch_size = 32, shuffle = True)